# Kaggle çalıştırıcı

Bu not defteri bilerek incedir: depoyu klonlar, bağımlılıkları kurar, veriyi yerine koyar ve bir betik çalıştırır. Mantık buraya yazılmaz; `src/` ve `scripts/` altında durur.

**Ayarlar (sağ panel):** Accelerator = GPU T4 x2, Internet = On. Input bölümüne veri setini ekle (`trgec-pilot-veri`).

`HF_TOKEN` yalnızca adaptör Hugging Face'e yüklenecekse gerekir; pilotta gerekmez.

In [ ]:
import os

GITHUB_USER = "aesivrioglu"
REPO = "Turkish_Correction_Model"

# Her satır scripts/run_pilot.py'nin bir çağrısıdır; sırayla çalışır. İkinci ve sonraki
# satırlarda --append olmalı (özet dosyası tek kalır). --clean-cache, o satırdaki modellerin
# indirilen ağırlıklarını iş bitince siler (disk için).
# Deneme koşusu: yalnızca en küçük aday. Sekiz adayın pilotu için --models kısmını sil ve --clean-cache ekle.
RUNS = [
    "--config configs/pilot.yaml --models ufakai/ufakzeka-1",
]

try:  # HF_TOKEN tanımlıysa kullanılır (indirme sınırları gevşer); yoksa devam edilir
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("HF_TOKEN tanımlı değil; açık modeller tokensız indirilecek.")

repo_url = f"https://github.com/{GITHUB_USER}/{REPO}.git"


In [ ]:
!git clone --depth 1 $repo_url /kaggle/working/repo
%cd /kaggle/working/repo
!git log -1 --oneline
!pip install -q -r requirements-kaggle.txt
!pip install -q -e .
!pip freeze | grep -i -E "^(torch|transformers|peft|accelerate|bitsandbytes|datasets)=="


In [ ]:
# Veri depoda yok; özel Kaggle veri setinden data/processed altına kopyalanır.
import glob, shutil

os.makedirs("data/processed", exist_ok=True)
found = glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
for path in found:
    shutil.copy(path, "data/processed/")
print(len(found), "dosya kopyalandı:", sorted(os.path.basename(p) for p in found))
assert any(p.endswith("gecturk_train_12k.jsonl") for p in found), "Veri seti eklenmemiş: sağ panelde Input > Add Input"


In [ ]:
!nvidia-smi


In [ ]:
for run in RUNS:
    print()
    print('########## run_pilot.py', run, flush=True)
    !python scripts/run_pilot.py $run


In [ ]:
# İndirilecek dosya: sağ panelde Output > pilot_outputs.zip
shutil.copy("outputs/pilot_outputs.zip", "/kaggle/working/pilot_outputs.zip")
print(open("outputs/pilot_summary.json", encoding="utf-8").read())


Bittiğinde `pilot_outputs.zip` dosyasını indir ve proje klasöründe `outputs/kaggle/` altına koy. Skorlama yerelde yapılır (`scripts/evaluate.py`); tahminler saklandığı için eğitimi yinelemek gerekmez.